# FireSight-IR | Module 3a (Fast) — Multi-Branch PINN Training

**Project:** FireSight-IR  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Platform:** Kaggle (recommended) or Google Colab free tier  

---

## Why the original was slow (and what this fixes)

| Problem | Root cause | Fix |
|---|---|---|
| 19 s/batch | HDF5 random-access reads per sample | Pre-cache to memory-mapped `.npy` (runs once) |
| 2+ hrs/epoch | DataLoader bottleneck, GPU starved | `.npy` memmap → 10–15× faster reads |
| Session timeout | 30 epochs × 2 hrs = 60 hrs needed | Target 15 min/epoch → full run in ~8 hrs |
| Resume broken | Scheduler not rebuilt correctly | Fixed `last_epoch` rebuild |

## Strategy

**Section 1:** Pre-cache (runs once, ~10 min). Reads HDF5 sequentially and writes flat `.npy` memmaps for patches, atm, srf, derived, labels.

**Section 2+:** Training. Each epoch reads from fast `.npy` memmaps — no HDF5 overhead.

**Kaggle setup:** Upload your Google Drive folder (or mount via KaggleHub). See Section 0.

## Expected speed
- Pre-cache: ~10 min (once only)
- Per epoch: ~12–18 min on T4/P100
- 30 epochs: ~6–9 hrs total (run in 2–3 Kaggle sessions with resume)

---
## Section 0 — Environment setup

### Option A — Kaggle (recommended)
1. Go to kaggle.com/code → New Notebook → Settings → Accelerator: GPU T4
2. Add your Google Drive data as a dataset (or use the Kaggle dataset upload)
3. Update `BASE_DIR` in Section 1 to point to your data

### Option B — Colab free tier
1. Runtime → Change runtime type → T4 GPU
2. Mount Drive (cell below)
3. Pre-cache runs once and saves to Drive — resumes automatically next session

In [ ]:
import os

# Detect environment
IN_COLAB  = 'google.colab' in str(__import__('sys').modules)
IN_KAGGLE = os.path.exists('/kaggle')

print(f'Environment: {"Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "Other"}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')

In [ ]:
!pip install torch tqdm scikit-learn matplotlib numpy h5py pyarrow -q

import os, json, time, warnings
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import matplotlib
matplotlib.use('Agg')  # works headless on Kaggle
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('[WARN] No GPU — training will be extremely slow')
print(f'PyTorch: {torch.__version__}')

---
## Section 1 — Paths and configuration

In [ ]:
# ── Paths — update BASE_DIR if you are on Kaggle ──────────────────────────────
if IN_KAGGLE:
    # Update this to wherever you uploaded your firesight-ir data on Kaggle
    BASE_DIR = '/kaggle/input/firesight-ir'
    OUT_DIR  = '/kaggle/working'  # Kaggle output directory
else:
    BASE_DIR = '/content/drive/MyDrive/firesight-ir'
    OUT_DIR  = BASE_DIR

ARCHIVE_PATH = f'{BASE_DIR}/data/processed/patches/firesight_patches.h5'
WEIGHTS_PATH = f'{BASE_DIR}/data/scalers/class_weights.json'
SPLIT_DIR    = f'{BASE_DIR}/data/splits'
MODEL_DIR    = f'{OUT_DIR}/models'
FIGURES_DIR  = f'{OUT_DIR}/figures'

# Pre-cache directory — saved to Drive/working so it persists across sessions
CACHE_DIR    = f'{OUT_DIR}/data/cache'

for d in [MODEL_DIR, FIGURES_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Feature dimensions (must match Module 2) ──────────────────────────────────
N_ATM      = 16
N_SRF      = 20
N_DERIVED  = 6
PATCH_SIZE = 32
N_CHANNELS = 4   # BT_I4, BT_I5, BTD, fire_mask
N_CLASSES  = 3

# ── Training hyperparameters ──────────────────────────────────────────────────
BATCH_SIZE   = 1024  # doubled vs original — GPU now has bandwidth headroom
N_EPOCHS     = 30
LR_MAX       = 3e-4
WEIGHT_DECAY = 1e-4
DROPOUT      = 0.3
NUM_WORKERS  = 4     # increase if you have more CPU cores (Kaggle: 4 is fine)

# ── Physics loss weights ──────────────────────────────────────────────────────
LAMBDA_BL  = 0.10
LAMBDA_DR  = 0.05
LAMBDA_TH  = 0.05
BT_I4_FIRE_MIN = 310.0
BTD_FIRE_MIN   =  10.0

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)

print('Configuration:')
print(f'  BASE_DIR  : {BASE_DIR}')
print(f'  CACHE_DIR : {CACHE_DIR}')
print(f'  MODEL_DIR : {MODEL_DIR}')
print(f'  Batch size: {BATCH_SIZE} | Epochs: {N_EPOCHS}')
print(f'  Workers   : {NUM_WORKERS}')

---
## Section 2 — Pre-cache HDF5 → numpy memmaps

This runs **once** and saves flat `.npy` files to Drive/working.  
On subsequent sessions the cell detects the cache and skips the extraction.

**Why this is 10–15× faster:**  
HDF5 random-access on Cloud Drive has ~10–50 ms latency per sample.  
A numpy memmap on local disk has ~0.1 ms. With batch=1024 this changes  
DataLoader from the bottleneck to negligible.

**Cache size:** ~4.5 GB total (patches: 4.4 GB, atm/srf/derived: ~50 MB)

In [ ]:
CACHE_FILES = {
    'patches' : f'{CACHE_DIR}/patches.npy',
    'atm'     : f'{CACHE_DIR}/atm.npy',
    'srf'     : f'{CACHE_DIR}/srf.npy',
    'derived' : f'{CACHE_DIR}/derived.npy',
    'labels'  : f'{CACHE_DIR}/labels.npy',
    'aux'     : f'{CACHE_DIR}/aux.npy',   # [BT_I4_center, BTD_center, beer_lambert]
}

def build_cache(archive_path, cache_files, chunk=4096):
    """
    Read HDF5 archive sequentially in chunks and write to numpy .npy files.
    Sequential HDF5 reads are ~10× faster than random-access per-sample reads.
    """
    print('Building numpy cache from HDF5 archive...')
    t0 = time.time()

    with h5py.File(archive_path, 'r') as hf:
        N = hf['labels'].shape[0]
        print(f'  Total samples: {N:,}')

        # Allocate output arrays
        # patches: (N, 4, 32, 32) — stack 4 channels
        patches_mm = np.lib.format.open_memmap(
            cache_files['patches'], mode='w+',
            dtype='float32', shape=(N, 4, PATCH_SIZE, PATCH_SIZE)
        )
        atm_mm     = np.lib.format.open_memmap(
            cache_files['atm'], mode='w+', dtype='float32', shape=(N, N_ATM)
        )
        srf_mm     = np.lib.format.open_memmap(
            cache_files['srf'], mode='w+', dtype='float32', shape=(N, N_SRF)
        )
        derived_mm = np.lib.format.open_memmap(
            cache_files['derived'], mode='w+', dtype='float32', shape=(N, N_DERIVED)
        )
        labels_mm  = np.lib.format.open_memmap(
            cache_files['labels'], mode='w+', dtype='uint8', shape=(N,)
        )
        aux_mm     = np.lib.format.open_memmap(
            cache_files['aux'], mode='w+', dtype='float32', shape=(N, 3)
        )

        has_derived = 'mlp_derived' in hf

        bar = tqdm(range(0, N, chunk), desc='Pre-caching', unit='chunk')
        for start in bar:
            end = min(start + chunk, N)
            sl  = slice(start, end)

            # Read channels
            bt_i4   = hf['cnn/bt_i4'][sl].astype(np.float32)
            bt_i5   = hf['cnn/bt_i5'][sl].astype(np.float32)
            bt_diff = hf['cnn/bt_diff'][sl].astype(np.float32)
            fm      = hf['cnn/fire_mask'][sl].astype(np.float32)

            # Normalise BT channels
            bt_i4_n  = (bt_i4  - 300.0) / 50.0
            bt_i5_n  = (bt_i5  - 290.0) / 20.0
            bt_diff_n = bt_diff / 40.0
            fm_n     = fm / 9.0

            patches_mm[sl] = np.stack([bt_i4_n, bt_i5_n, bt_diff_n, fm_n], axis=1)

            atm  = hf['mlp_atm'][sl].astype(np.float32)
            srf  = hf['mlp_srf'][sl].astype(np.float32)
            derived = hf['mlp_derived'][sl].astype(np.float32) if has_derived \
                      else np.zeros((end-start, N_DERIVED), dtype=np.float32)
            lbl  = hf['labels'][sl]

            # Physics aux: [BT_I4_center, BTD_center, beer_lambert_proxy]
            bt_i4_center = bt_i4[:, 16, 16]          # raw K
            btd_center   = hf['cnn/bt_diff'][sl][:, 16, 16].astype(np.float32)
            beer_lambert = atm[:, 14] if atm.shape[1] > 14 else np.full(end-start, 0.72)

            # NaN → 0 for safety
            atm     = np.nan_to_num(atm,     nan=0.0)
            srf     = np.nan_to_num(srf,     nan=0.0)
            derived = np.nan_to_num(derived, nan=0.0)

            atm_mm[sl]     = atm
            srf_mm[sl]     = srf
            derived_mm[sl] = derived
            labels_mm[sl]  = lbl
            aux_mm[sl]     = np.stack([bt_i4_center, btd_center, beer_lambert], axis=1)

            bar.set_postfix({'written': f'{end:,}'})

        # Flush
        for mm in [patches_mm, atm_mm, srf_mm, derived_mm, labels_mm, aux_mm]:
            mm.flush()

    elapsed = time.time() - t0
    print(f'\n✓ Cache built in {elapsed/60:.1f} min')
    sizes = {k: f'{os.path.getsize(v)/1e9:.2f} GB'
             for k, v in cache_files.items() if os.path.exists(v)}
    print(f'  Cache sizes: {sizes}')


# ── Run pre-cache (skips if already done) ────────────────────────────────────
all_cached = all(os.path.exists(p) for p in CACHE_FILES.values())

if all_cached:
    print('✓ Cache already exists — skipping extraction')
    print(f'  Cache files:')
    for k, p in CACHE_FILES.items():
        sz = os.path.getsize(p) / 1e9
        print(f'    {k:10s}: {sz:.2f} GB')
else:
    missing = [k for k, p in CACHE_FILES.items() if not os.path.exists(p)]
    print(f'Missing cache files: {missing}')
    build_cache(ARCHIVE_PATH, CACHE_FILES)

---
## Section 3 — Fast Dataset class (reads from memmap)

In [ ]:
class FireSightFastDataset(Dataset):
    """
    Reads from pre-cached numpy memmaps — no HDF5 overhead.
    ~10-15× faster than the HDF5-based dataset.

    Augmentation (training only):
      - Random 90° rotations (4 orientations)
      - Random horizontal flip
    """

    def __init__(self, cache_files, indices, augment=False):
        self.indices = np.sort(indices)
        self.augment = augment

        # Open memmaps read-only — shared across workers, zero copy
        self.patches = np.load(cache_files['patches'], mmap_mode='r')
        self.atm     = np.load(cache_files['atm'],     mmap_mode='r')
        self.srf     = np.load(cache_files['srf'],     mmap_mode='r')
        self.derived = np.load(cache_files['derived'], mmap_mode='r')
        self.labels  = np.load(cache_files['labels'],  mmap_mode='r')
        self.aux     = np.load(cache_files['aux'],     mmap_mode='r')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])

        patch   = self.patches[idx].copy()   # (4, 32, 32)
        atm     = self.atm[idx].copy()
        srf     = self.srf[idx].copy()
        derived = self.derived[idx].copy()
        label   = int(self.labels[idx])
        aux     = self.aux[idx].copy()

        # Augmentation (training only)
        if self.augment:
            k = np.random.randint(0, 4)
            if k > 0:
                patch = np.rot90(patch, k, axes=(1, 2)).copy()
            if np.random.rand() > 0.5:
                patch = np.flip(patch, axis=2).copy()

        return (
            torch.from_numpy(patch),
            torch.from_numpy(atm),
            torch.from_numpy(srf),
            torch.from_numpy(derived),
            torch.tensor(label, dtype=torch.long),
            torch.from_numpy(aux),
        )


print('FireSightFastDataset defined.')

---
## Section 4 — Model architecture

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.2):
        super().__init__()
        self.fc1  = nn.Linear(in_dim,  out_dim)
        self.fc2  = nn.Linear(out_dim, out_dim)
        self.bn1  = nn.BatchNorm1d(out_dim)
        self.bn2  = nn.BatchNorm1d(out_dim)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        identity = self.proj(x)
        out = F.relu(self.bn1(self.fc1(x)))
        out = self.drop(out)
        out = self.bn2(self.fc2(out))
        return F.relu(out + identity)


class CNNBranch(nn.Module):
    """4-channel 32×32 patch → 128-dim embedding."""
    def __init__(self, in_channels=4, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(True),
            nn.Conv2d(32,          32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(True),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(32,          64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.Conv2d(64,          64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(64,          128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2),
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.gap(self.encoder(x)).flatten(1))


class FireSightPINN(nn.Module):
    """
    Multi-branch PINN: CNN + MLP-atm + MLP-srf + MLP-derived → 3-class
    + transmittance head for Beer-Lambert physics loss.
    """
    def __init__(self, n_atm=16, n_srf=20, n_derived=6,
                 n_classes=3, dropout=0.3):
        super().__init__()
        self.cnn_branch     = CNNBranch(4, dropout)
        self.atm_branch     = nn.Sequential(ResidualBlock(n_atm, 64), ResidualBlock(64, 32))
        self.srf_branch     = nn.Sequential(ResidualBlock(n_srf, 64), ResidualBlock(64, 32))
        self.derived_branch = nn.Sequential(
            nn.Linear(n_derived, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 16),        nn.BatchNorm1d(16), nn.ReLU(),
        )
        fusion_in = 128 + 32 + 32 + 16  # 208
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout),
        )
        self.classifier        = nn.Linear(64, n_classes)
        self.transmittance_head = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, patch, atm, srf, derived):
        fused    = torch.cat([
            self.cnn_branch(patch),
            self.atm_branch(atm),
            self.srf_branch(srf),
            self.derived_branch(derived),
        ], dim=1)
        features = self.fusion(fused)
        return self.classifier(features), self.transmittance_head(features)


# Sanity check
model = FireSightPINN(N_ATM, N_SRF, N_DERIVED, N_CLASSES, DROPOUT).to(device)
total = sum(p.numel() for p in model.parameters())
print(f'FireSightPINN: {total:,} parameters')
with torch.no_grad():
    _p = torch.randn(4, N_CHANNELS, PATCH_SIZE, PATCH_SIZE).to(device)
    _a = torch.randn(4, N_ATM).to(device)
    _s = torch.randn(4, N_SRF).to(device)
    _d = torch.randn(4, N_DERIVED).to(device)
    _l, _t = model(_p, _a, _s, _d)
    print(f'Dry run: logits={_l.shape}, transmittance={_t.shape} ✓')

---
## Section 5 — Physics-informed loss

In [ ]:
class FireSightPINNLoss(nn.Module):
    def __init__(self, class_weights, lambda_bl=0.1, lambda_dr=0.05,
                 lambda_th=0.05, bt_min=310.0, btd_min=10.0, device='cpu'):
        super().__init__()
        self.lambda_bl = lambda_bl
        self.lambda_dr = lambda_dr
        self.lambda_th = lambda_th
        self.bt_min    = bt_min
        self.btd_min   = btd_min
        w = torch.tensor(class_weights, dtype=torch.float32).to(device)
        self.ce  = nn.CrossEntropyLoss(weight=w)
        self.mse = nn.MSELoss()

    def forward(self, logits, transmittance, labels, aux):
        L_ce   = self.ce(logits, labels)
        L_bl   = self.mse(transmittance, aux[:, 2:3])
        probs  = F.softmax(logits, dim=1)
        p_wf   = probs[:, 1]
        L_dr   = (p_wf * (aux[:, 0] < self.bt_min).float()).mean()
        L_th   = (p_wf * (aux[:, 1] < self.btd_min).float()).mean()
        total  = L_ce + self.lambda_bl*L_bl + self.lambda_dr*L_dr + self.lambda_th*L_th
        return total, {'ce': L_ce.item(), 'bl': L_bl.item(),
                       'dr': L_dr.item(), 'th': L_th.item()}


print('FireSightPINNLoss defined.')

---
## Section 6 — Load splits and build DataLoaders

In [ ]:
train_idx = np.load(f'{SPLIT_DIR}/train_index.npy')
val_idx   = np.load(f'{SPLIT_DIR}/val_index.npy')
test_idx  = np.load(f'{SPLIT_DIR}/test_index.npy')

print(f'Splits — Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}')

with open(WEIGHTS_PATH) as f:
    cw = json.load(f)
if isinstance(cw, list):
    class_weights = cw
elif isinstance(cw, dict):
    class_weights = [float(cw.get(str(i), [18.98,1.0,27.02][i])) for i in range(3)]
else:
    class_weights = [18.98, 1.0, 27.02]
print(f'Class weights: {[round(w,2) for w in class_weights]}')

train_ds = FireSightFastDataset(CACHE_FILES, train_idx, augment=True)
val_ds   = FireSightFastDataset(CACHE_FILES, val_idx,   augment=False)
test_ds  = FireSightFastDataset(CACHE_FILES, test_idx,  augment=False)

pin = device.type == 'cuda'
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,  shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin,
                          persistent_workers=True, prefetch_factor=3)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin,
                          persistent_workers=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin)

print(f'DataLoaders ready.')
print(f'  Train batches: {len(train_loader):,} (batch={BATCH_SIZE})')
print(f'  Val batches  : {len(val_loader):,}')

# Benchmark one batch to confirm speed improvement
t0 = time.time()
batch = next(iter(train_loader))
t1 = time.time()
print(f'  First batch load time: {t1-t0:.2f}s')
print(f'  Batch shapes: patch={batch[0].shape}, atm={batch[1].shape}, label={batch[4].shape}')

---
## Section 7 — Training (with progress bar and resume)

In [ ]:
BEST_PATH  = f'{MODEL_DIR}/firesight_pinn_best.pt'
FINAL_PATH = f'{MODEL_DIR}/firesight_pinn_final.pt'
LOG_PATH   = f'{MODEL_DIR}/training_log.json'

# ── Init or resume ────────────────────────────────────────────────────────────
best_val_loss = float('inf')
training_log  = []
start_epoch   = 0

model     = FireSightPINN(N_ATM, N_SRF, N_DERIVED, N_CLASSES, DROPOUT).to(device)
criterion = FireSightPINNLoss(
    class_weights, LAMBDA_BL, LAMBDA_DR, LAMBDA_TH,
    BT_I4_FIRE_MIN, BTD_FIRE_MIN, device
)

if os.path.exists(BEST_PATH):
    ckpt = torch.load(BEST_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    best_val_loss = ckpt.get('val_loss', float('inf'))
    start_epoch   = ckpt.get('epoch', 0)
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH) as f:
            training_log = json.load(f)
    print(f'Resumed from epoch {start_epoch} | best val loss {best_val_loss:.4f}')
else:
    print('Starting fresh training')

optimizer = AdamW(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)

# Rebuild scheduler — skip steps already completed
completed_steps = start_epoch * len(train_loader)
scheduler = OneCycleLR(
    optimizer,
    max_lr          = LR_MAX,
    steps_per_epoch = len(train_loader),
    epochs          = N_EPOCHS,
    pct_start       = 0.1,
    anneal_strategy = 'cos',
    last_epoch      = completed_steps - 1,
)

print(f'Training epochs {start_epoch+1}–{N_EPOCHS}')
print(f'Expected time: ~{len(train_loader)*2//60}–{len(train_loader)*3//60} min/epoch on T4\n')

In [ ]:
# ── Main training loop ────────────────────────────────────────────────────────
for epoch in range(start_epoch, N_EPOCHS):
    t0 = time.time()

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0
    comp_acc   = {'ce':0,'bl':0,'dr':0,'th':0}
    correct = 0
    n_total = 0

    bar = tqdm(
        train_loader,
        desc  = f'Ep {epoch+1:02d}/{N_EPOCHS}',
        ncols = 110,
        leave = True,
    )
    for patch, atm, srf, derived, labels, aux in bar:
        patch   = patch.to(device,   non_blocking=True)
        atm     = atm.to(device,     non_blocking=True)
        srf     = srf.to(device,     non_blocking=True)
        derived = derived.to(device, non_blocking=True)
        labels  = labels.to(device,  non_blocking=True)
        aux     = aux.to(device,     non_blocking=True)

        optimizer.zero_grad()
        logits, trans = model(patch, atm, srf, derived)
        loss, comp    = criterion(logits, trans, labels, aux)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        bs = len(labels)
        total_loss += loss.item() * bs
        for k in comp_acc: comp_acc[k] += comp[k] * bs
        correct   += (logits.argmax(1) == labels).sum().item()
        n_total   += bs

        bar.set_postfix(
            loss=f'{loss.item():.3f}',
            acc =f'{correct/n_total:.3f}',
            lr  =f'{scheduler.get_last_lr()[0]:.1e}',
        )

    train_loss = total_loss / n_total
    train_acc  = correct / n_total
    avg_comp   = {k: v/n_total for k, v in comp_acc.items()}

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    val_total = 0
    vp, vl    = [], []

    with torch.no_grad():
        for patch, atm, srf, derived, labels, aux in val_loader:
            patch   = patch.to(device, non_blocking=True)
            atm     = atm.to(device,   non_blocking=True)
            srf     = srf.to(device,   non_blocking=True)
            derived = derived.to(device, non_blocking=True)
            labels  = labels.to(device, non_blocking=True)
            aux     = aux.to(device,   non_blocking=True)
            logits, trans = model(patch, atm, srf, derived)
            loss, _   = criterion(logits, trans, labels, aux)
            val_total += loss.item() * len(labels)
            vp.append(logits.argmax(1).cpu().numpy())
            vl.append(labels.cpu().numpy())

    val_loss   = val_total / len(val_idx)
    val_preds  = np.concatenate(vp)
    val_labels = np.concatenate(vl)
    val_acc    = (val_preds == val_labels).mean()

    pca = {}
    for cls, name in [(0,'nf'),(1,'wf'),(2,'fa')]:
        m = val_labels == cls
        pca[name] = float((val_preds[m] == cls).mean()) if m.sum()>0 else 0.0

    elapsed = time.time() - t0

    print(f'  → val loss={val_loss:.4f} acc={val_acc:.3f} | '
          f'nf={pca["nf"]:.3f} wf={pca["wf"]:.3f} fa={pca["fa"]:.3f} | '
          f'{elapsed:.0f}s')

    # ── Save checkpoint ───────────────────────────────────────────────────────
    epoch_log = {
        'epoch': epoch+1, 'train_loss': round(train_loss,4),
        'train_acc': round(train_acc,4), 'val_loss': round(val_loss,4),
        'val_acc': round(val_acc,4), 'elapsed_s': round(elapsed,1),
        'loss_components': {k: round(v,5) for k, v in avg_comp.items()},
        'per_class_val_acc': {k: round(v,4) for k, v in pca.items()},
    }
    training_log.append(epoch_log)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch+1, 'model_state_dict': model.state_dict(),
            'val_loss': val_loss, 'val_acc': val_acc,
            'class_weights': class_weights,
        }, BEST_PATH)
        print(f'  ✓ Best model saved (val_loss={val_loss:.4f})')

    with open(LOG_PATH, 'w') as f:
        json.dump(training_log, f, indent=2)

torch.save({'epoch': N_EPOCHS, 'model_state_dict': model.state_dict()}, FINAL_PATH)
print(f'\n✓ Training complete | best val loss: {best_val_loss:.4f}')

---
## Section 8 — Evaluation

In [ ]:
ckpt = torch.load(BEST_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Best checkpoint: epoch {ckpt["epoch"]} | val_loss={ckpt["val_loss"]:.4f}')

@torch.no_grad()
def evaluate(loader, name):
    model.eval()
    preds, lbls, probs_list = [], [], []
    for patch, atm, srf, derived, labels, aux in loader:
        logits, _ = model(
            patch.to(device), atm.to(device),
            srf.to(device), derived.to(device)
        )
        preds.append(logits.argmax(1).cpu().numpy())
        lbls.append(labels.numpy())
        probs_list.append(F.softmax(logits, dim=1).cpu().numpy())
    preds = np.concatenate(preds)
    lbls  = np.concatenate(lbls)
    probs = np.concatenate(probs_list)
    print(f'\n=== {name} ===')
    print(classification_report(lbls, preds,
          target_names=['non-fire','wildfire','false-alarm'], digits=4))
    return preds, lbls, probs

test_preds, test_labels, test_probs = evaluate(test_loader, 'Test set (2018-2022 held-out 20%)')
val_preds,  val_labels,  val_probs  = evaluate(val_loader,  'Val set (2023 fully held-out)')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
with open(LOG_PATH) as f:
    log = json.load(f)

ep  = [e['epoch']      for e in log]
trl = [e['train_loss'] for e in log]
vll = [e['val_loss']   for e in log]
tra = [e['train_acc']  for e in log]
vla = [e['val_acc']    for e in log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0a0a0a')
fig.patch.set_facecolor('#0a0a0a')

for ax in (ax1, ax2):
    ax.set_facecolor('#0a0a0a')
    ax.tick_params(colors='#666')
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.set_xlabel('Epoch', color='#888')

ax1.plot(ep, trl, color='#4FC3F7', label='Train')
ax1.plot(ep, vll, color='#FF7043', label='Val (2023)')
ax1.set_title('Loss', color='white')
ax1.legend(facecolor='#1a1a1a', labelcolor='white')

ax2.plot(ep, tra, color='#4FC3F7', label='Train')
ax2.plot(ep, vla, color='#FF7043', label='Val (2023)')
ax2.set_title('Accuracy', color='white')
ax2.set_ylim(0, 1)
ax2.legend(facecolor='#1a1a1a', labelcolor='white')

fig.suptitle('FireSight-IR | Module 3a — Training Curves', color='white', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03a_training_curves.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print(f'Figure saved → {FIGURES_DIR}/03a_training_curves.png')

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(13, 6), facecolor='#0a0a0a')
fig.patch.set_facecolor('#0a0a0a')
class_names = ['Non-fire', 'Wildfire', 'False-alarm']

for ax, (preds, labels, title) in zip(axes, [
    (test_preds, test_labels, 'Test set (2018-2022)'),
    (val_preds,  val_labels,  'Val set (2023)'),
]):
    cm = confusion_matrix(labels, preds, normalize='true')
    im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(class_names, rotation=30, ha='right', color='white', fontsize=9)
    ax.set_yticklabels(class_names, color='white', fontsize=9)
    ax.set_xlabel('Predicted', color='#888')
    ax.set_ylabel('True',      color='#888')
    ax.set_title(title, color='white')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{cm[i,j]:.3f}', ha='center', va='center',
                    color='white' if cm[i,j] < 0.5 else 'black', fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle('FireSight-IR | Module 3a — Confusion Matrices', color='white', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03a_confusion_matrix.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print('Confusion matrices saved.')

In [ ]:
print('=' * 55)
print('  FireSight-IR | Module 3a Complete')
print('=' * 55)
print(f'  Architecture : Multi-branch PINN ({sum(p.numel() for p in model.parameters()):,} params)')
print(f'  Training     : {len(training_log)} epochs, OneCycleLR, AdamW')
print(f'  Best val loss: {best_val_loss:.4f}')
print(f'  Val accuracy : {ckpt["val_acc"]:.4f} (2023 held-out)')
print()
print(f'  Best model  : {BEST_PATH}')
print(f'  Training log: {LOG_PATH}')
print()
print(f'  Next: Module 3b — False-alarm discriminator')
print('=' * 55)